# OSNet-AIN x1.0 — вариант 2: Optuna, P=16/K=2, BNNeck и SupCon

Этот notebook создаёт **новый development-эксперимент**, не изменяя вариант 1 и активный MVP.

Порядок работы:

1. smoke test новой модели и loss;
2. 16 Optuna trials по 8 эпох на внутреннем разбиении;
3. ранняя остановка слабых trials не раньше четвёртой эпохи;
4. продолжение четырёх лучших конфигураций до 20 эпох;
5. независимое обучение двух финалистов до 30 эпох с тремя seed;
6. выбор по среднему inner-validation mAP и обучение победителя на всех 925 train identity;
7. оценка на прежних calibration/validation и сохранение важных checkpoint/ONNX.

`test_query.csv` и `test_gallery.csv` не используются при обучении, HPO или выборе checkpoint.

## 1. Окружение

Из корня репозитория:

```bash
uv pip install --python .venv/bin/python -r requirements-train.txt
uv pip check --python .venv/bin/python
.venv/bin/python -m jupyter lab
```

Один trial ограничен 8 эпохами. Median pruner может остановить слабый запуск только после четвёртой эпохи. Затем top-4 продолжаются до 20 эпох, а top-2 проверяются с тремя seed по 30 эпох. На MPS полный `Run All` займёт несколько часов.

In [ ]:
import gc
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
assert (ROOT / 'backend').exists(), 'Не найден корень Car-classification-MSK'
os.chdir(ROOT)

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch
from PIL import Image
from torch.nn import functional as F

from backend.core import DATASET, STOCK_MODEL, bbox, preprocess, sha256
from backend.evaluate import write_json
from training.hpo import (
    ExperimentConfig, config_from_trial, continue_top_candidates, experiment_losses, fit_selected,
    initialize_experiment, load_hpo_study, make_optimizer, prepare_experiment,
    run_finalist_seeds, run_hpo, set_epoch_learning_rates, split_hpo_identities,
)
from training.osnet import export_encoder_onnx
from training.pipeline import ensure_splits, select_device, set_seed, split_summary

VARIANT_DIR = ROOT / 'OSNet-AIN-x1.0/variant_02_hpo_bnneck_supcon'
WEIGHTS_DIR = VARIANT_DIR / 'weights'
RESULTS_DIR = VARIANT_DIR / 'results'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_TRIALS = 16
TRIAL_EPOCHS = 8
TOP_CANDIDATES = 4
PROMOTION_EPOCHS = 20
FINALISTS = 2
FINALIST_EPOCHS = 30
FINALIST_SEEDS = (20260915, 20260916, 20260917)
OUTER_EPOCHS = 30
RUN_HPO = True
RUN_STAGE2 = True
RUN_STAGE3 = True
RUN_SELECTED_TRAINING = True
RUN_EXPORT = True
RUN_NAME = 'selected_run_02'  # Измените имя перед новым независимым полным запуском.

SELECTED_WEIGHTS = WEIGHTS_DIR / RUN_NAME
SELECTED_RESULTS = RESULTS_DIR / RUN_NAME
DEVICE = select_device()
set_seed(20260915)
print('python:', sys.executable)
print('device:', DEVICE, '| torch:', torch.__version__)
print('variant:', VARIANT_DIR)

## 2. Разбиения без утечки

Внешнее разбиение остаётся тем же, что в варианте 1: 925 train, 307 calibration и 309 validation identity.

Для Optuna 925 train identity дополнительно делятся на:

- 80% HPO-train;
- 20% HPO-validation.

Внешние calibration и validation не участвуют в поиске гиперпараметров. Они используются только после завершения HPO для выбранной конфигурации.

In [ ]:
rows, split = ensure_splits(DATASET)
print(json.dumps(split_summary(rows, split), ensure_ascii=False, indent=2))
hpo_train_ids, hpo_validation_ids = split_hpo_identities(split['identities']['train'])
print('HPO train identities:', len(hpo_train_ids))
print('HPO validation identities:', len(hpo_validation_ids))
assert set(hpo_train_ids).isdisjoint(hpo_validation_ids)
assert set(hpo_train_ids + hpo_validation_ids) == set(split['identities']['train'])

## 3. Smoke test: P=16/K=2, разные камеры, BNNeck и SupCon

Вариант 2 использует 16 identity по 2 изображения. Это даёт каждому anchor 15 negative identity вместо 7 в варианте 1. Sampler по возможности берёт два positive с разных камер. `camera_id` используется только для формирования train-batch и не подаётся модели.

Encoder и случайно созданная голова обучаются с разными learning rate. BNNeck и тип metric loss затем выбирает Optuna.

In [ ]:
SMOKE_CONFIG = ExperimentConfig(epochs=1, identities_per_batch=16, images_per_identity=2,
                                metric_loss='supcon', use_bnneck=True)
smoke_rows, smoke_labels, smoke_sampler, smoke_loader = prepare_experiment(
    rows, hpo_train_ids, SMOKE_CONFIG, DATASET
)
indices = next(iter(smoke_sampler))
for label in {smoke_rows[index]['label'] for index in indices}:
    selected = [smoke_rows[index] for index in indices if smoke_rows[index]['label'] == label]
    assert len(selected) == 2
    assert len({item['camera_id'] for item in selected}) == 2

clean, robust, labels, _ = next(iter(smoke_loader))
assert clean.shape == robust.shape == (32, 3, 208, 208)
assert len(set(labels.tolist())) == 16
assert all(labels.tolist().count(label) == 2 for label in set(labels.tolist()))

smoke_model, loaded_tensors = initialize_experiment(len(smoke_labels), SMOKE_CONFIG, DEVICE, STOCK_MODEL)
smoke_optimizer = make_optimizer(smoke_model, SMOKE_CONFIG)
lrs = set_epoch_learning_rates(smoke_optimizer, SMOKE_CONFIG, 0, 1)
smoke_optimizer.zero_grad(set_to_none=True)
losses = experiment_losses(smoke_model, clean.to(DEVICE), robust.to(DEVICE), labels.to(DEVICE), SMOKE_CONFIG)
losses['loss'].backward()
smoke_optimizer.step()
assert all(torch.isfinite(value) for value in losses.values())
print('loaded stock tensors:', loaded_tensors)
print('learning rates:', lrs)
print({name: round(float(value.detach()), 6) for name, value in losses.items()})

del smoke_model, smoke_optimizer, smoke_loader, clean, robust, labels, losses
gc.collect()
if DEVICE.type == 'mps': torch.mps.empty_cache()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 4. Optuna HPO — 16 trials

Поиск фиксирует `P=16, K=2` и перебирает:

- Triplet или Supervised Contrastive loss;
- BNNeck включён/выключен;
- encoder LR `5e-5 … 3e-4`;
- множитель LR головы `5` или `10`;
- weight decay `1e-5 … 1e-3`;
- вес metric loss;
- triplet margin, если выбран Triplet;
- вес consistency loss.

Study хранится в `results/optuna.sqlite3`. Повторный запуск продолжает study только до общего числа `TARGET_TRIALS`, а не добавляет ещё 16. Слабый trial разрешено прервать только после четырёх законченных эпох.

In [ ]:
if RUN_HPO:
    study = run_hpo(rows, split, DEVICE, RESULTS_DIR, WEIGHTS_DIR,
                    target_trials=TARGET_TRIALS, trial_epochs=TRIAL_EPOCHS, dataset=DATASET)
else:
    study = load_hpo_study(RESULTS_DIR)

print('best trial:', study.best_trial.number)
print('inner-validation mAP:', study.best_value)
print(json.dumps(study.best_trial.params, ensure_ascii=False, indent=2))

In [ ]:
completed = [trial for trial in study.trials if trial.value is not None]
completed = sorted(completed, key=lambda trial: trial.value, reverse=True)
for trial in completed[:10]:
    print(f"trial={trial.number:02d} | state={trial.state.name:8s} | mAP={trial.value:.4f} | "
          f"metric={trial.params.get('metric_loss')} | bnneck={trial.params.get('use_bnneck')} | "
          f"lr={trial.params.get('encoder_lr'):.2e}")

if RUN_STAGE2:
    stage2_summary = continue_top_candidates(
        rows, split, study, DEVICE, RESULTS_DIR, WEIGHTS_DIR,
        top_k=TOP_CANDIDATES, target_epochs=PROMOTION_EPOCHS, dataset=DATASET,
    )
else:
    stage2_summary = json.loads((RESULTS_DIR / 'stage2_top4_summary.json').read_text())
print('top-4 after 20 epochs:', [(item['trial_number'], round(item['best_mAP'], 4))
                                    for item in stage2_summary['candidates']])

if RUN_STAGE3:
    stage3_summary = run_finalist_seeds(
        rows, split, study, stage2_summary, DEVICE, RESULTS_DIR, WEIGHTS_DIR,
        top_k=FINALISTS, target_epochs=FINALIST_EPOCHS, seeds=FINALIST_SEEDS, dataset=DATASET,
    )
else:
    stage3_summary = json.loads((RESULTS_DIR / 'stage3_top2_seeds_summary.json').read_text())
winner_number = stage3_summary['best_trial_number']
winner_trial = next(trial for trial in study.trials if trial.number == winner_number)
print('winner trial by mean seed mAP:', winner_number)
print(json.dumps(stage3_summary['finalists'], ensure_ascii=False, indent=2))

## 5. Полное development-обучение победившей конфигурации

Победитель выбирается по среднему лучшему mAP на трёх seed. После этого конфигурация заново инициализируется из стокового ONNX и обучается на всех 925 train identity 30 эпох. Внешние calibration/validation впервые используются только здесь.

Checkpoint policy:

- `last.pt` обновляется после каждой завершённой эпохи;
- `best_map.pt`, `best_f1.pt`, `best_tnr.pt` хранят лидеров отдельных метрик;
- `epoch_XX.pt` сохраняется один раз, если эпоха улучшила хотя бы одну из этих метрик;
- `checkpoint_index.json` связывает метрику с номером эпохи.

Так mAP и качество отказа больше не смешиваются в один неявный критерий.

In [ ]:
SELECTED_CONFIG = config_from_trial(winner_trial, epochs=OUTER_EPOCHS, seed=FINALIST_SEEDS[0])
print(json.dumps(asdict(SELECTED_CONFIG), ensure_ascii=False, indent=2))
write_json(RESULTS_DIR / 'selected_config.json', asdict(SELECTED_CONFIG))

if RUN_SELECTED_TRAINING:
    assert not (SELECTED_RESULTS / 'history.json').exists(), (
        f'{RUN_NAME} уже содержит history.json. Измените RUN_NAME, чтобы не перезаписать эксперимент.'
    )
    set_seed(SELECTED_CONFIG.seed)
    selected_rows, selected_labels, selected_sampler, selected_loader = prepare_experiment(
        rows, split['identities']['train'], SELECTED_CONFIG, DATASET
    )
    selected_model, loaded_tensors = initialize_experiment(
        len(selected_labels), SELECTED_CONFIG, DEVICE, STOCK_MODEL
    )
    history, checkpoint_index = fit_selected(
        selected_model, rows, split, selected_loader, selected_sampler, DEVICE,
        SELECTED_CONFIG, SELECTED_WEIGHTS, SELECTED_RESULTS,
    )
else:
    history_path = SELECTED_RESULTS / 'history.json'
    history = json.loads(history_path.read_text()) if history_path.exists() else []
    checkpoint_index_path = SELECTED_RESULTS / 'checkpoint_index.json'
    checkpoint_index = json.loads(checkpoint_index_path.read_text()) if checkpoint_index_path.exists() else None
    print('Полное обучение выключено; загружены существующие результаты:', bool(history))

## 6. Графики и отдельный выбор checkpoint

Основной retrieval checkpoint — `best_map.pt`. Для продуктового режима отказа отдельно сравниваются `best_f1.pt` и `best_tnr.pt`; они могут оказаться другими эпохами.

In [ ]:
if history:
    epochs = [item['epoch'] for item in history]
    maps = [item['validation']['mAP'] for item in history]
    f1 = [item['validation']['candidate_F1'] for item in history]
    tnr = [item['validation']['TNR'] for item in history]
    losses = [item['train']['loss'] for item in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, losses, marker='o'); axes[0].set_title('Train loss'); axes[0].set_xlabel('epoch')
    axes[1].plot(epochs, maps, marker='o', label='mAP')
    axes[1].plot(epochs, f1, marker='o', label='Candidate F1')
    axes[1].plot(epochs, tnr, marker='o', label='TNR')
    axes[1].axhline(.773992168, color='gray', linestyle='--', label='variant 1 mAP')
    axes[1].set_title('Outer validation'); axes[1].set_xlabel('epoch'); axes[1].legend()
    plt.tight_layout()
    print(json.dumps(checkpoint_index, ensure_ascii=False, indent=2))

## 7. Экспорт лучшего mAP-checkpoint в ONNX

Экспорт включает BNNeck, если он выбран Optuna. После экспорта PyTorch и ONNX сравниваются на реальном BBox-кропе. Новый ONNX не подключается к MVP автоматически.

In [ ]:
if RUN_EXPORT:
    checkpoint_path = SELECTED_WEIGHTS / 'best_map.pt'
    assert checkpoint_path.exists(), 'Сначала выполните полное development-обучение'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    export_config = ExperimentConfig(**checkpoint['config'])
    export_model, _ = initialize_experiment(len(split['identities']['train']), export_config, torch.device('cpu'))
    export_model.load_state_dict(checkpoint['model'])
    export_model.eval()

    onnx_path = SELECTED_WEIGHTS / 'osnet_best_map.onnx'
    export_encoder_onnx(export_model.inference_module(), onnx_path, 'cpu')

    sample = next(row for row in rows if row['vehicle_id'] in set(split['identities']['validation']))
    with Image.open(DATASET / 'images' / f"{sample['image_id']}.jpg") as image:
        array = preprocess(image, bbox(sample))[None]
    with torch.inference_mode():
        pytorch_output = export_model.embedding(torch.from_numpy(array)).numpy()
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    onnx_output = session.run(['output'], {session.get_inputs()[0].name: array})[0]
    max_difference = float(np.max(np.abs(pytorch_output - onnx_output)))
    cosine = float(F.cosine_similarity(torch.from_numpy(pytorch_output), torch.from_numpy(onnx_output)).item())
    parity = {'max_absolute_difference': max_difference, 'cosine_similarity': cosine}
    assert cosine > .99999 and max_difference < 1e-3, parity

    summary = {
        'run_name': RUN_NAME,
        'checkpoint': checkpoint_path.name,
        'checkpoint_epoch': checkpoint['epoch'],
        'config': checkpoint['config'],
        'threshold': checkpoint['threshold'],
        'calibration': checkpoint['calibration'],
        'validation': checkpoint['validation'],
        'onnx': str(onnx_path.relative_to(ROOT)),
        'onnx_sha256': sha256(onnx_path),
        'parity': parity,
    }
    write_json(SELECTED_RESULTS / 'summary.json', summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. Что прислать после обучения

Для анализа достаточно файлов:

- `results/study_summary.json`;
- `results/stage2_top4_summary.json`;
- `results/stage3_top2_seeds_summary.json`;
- `results/selected_config.json`;
- `results/<RUN_NAME>/history.json`;
- `results/<RUN_NAME>/checkpoint_index.json`;
- `results/<RUN_NAME>/summary.json`;
- при необходимости — output ячеек notebook.

После сравнения с вариантом 1 отдельно решим, какой checkpoint подключать к MVP и стоит ли переходить к финальному обучению на train+validation.